In [60]:
import pandas as pd
from typing import Union, List, Dict
import json
import math

In [61]:
def dcg(relevances: List[float]) -> float:
    return sum((2**rel - 1) / math.log2(idx + 2) for idx, rel in enumerate(relevances))

def ndcg_at_k(retrieved: List[str], qrels: Dict[str, float], k: int = 10) -> float:
    # 截断
    retrieved = retrieved[:k]

    # 将相关性转为 float
    rels = [float(qrels.get(doc_id, 0)) for doc_id in retrieved]
    dcg_val = dcg(rels)

    # 理想 DCG
    ideal_rels = sorted([float(s) for s in qrels.values()], reverse=True)[:k]
    idcg_val = dcg(ideal_rels)

    return dcg_val / idcg_val if idcg_val > 0 else 0.0

def mean_ndcg(retrieved_dict: Dict[str, List[str]], all_qrels: Dict[str, Dict[str, float]], k: int = 10) -> float:
    """
    计算所有 query 的平均 NDCG@k

    参数：
        retrieved_dict: {query_id: [doc_id1, doc_id2, ...], ...}
        all_qrels: {query_id: {doc_id: score, ...}, ...}
        k: 截断位置，默认 10

    返回：
        float: 平均 NDCG@k
    """
    ndcg_scores = []
    for qid, retrieved in retrieved_dict.items():
        # print("Evaluating query:", qid)
        # print("Retrieved documents:", retrieved)
        qrels = all_qrels.get(qid, {})
        # print("Qrels:", qrels)
        if not qrels:
            continue  # 如果没有 qrels，跳过该 query
        ndcg_scores.append(ndcg_at_k(retrieved, qrels, k))
    
    return sum(ndcg_scores) / len(ndcg_scores) if ndcg_scores else 0.0

In [62]:
def load_qrels(qrels_paths: Union[str, List[str]]) -> dict:
    """
    加载一个或多个 qrels 文件并转换为字典格式。

    参数：
        qrels_paths (str 或 List[str]): qrels 文件路径或路径列表，文件为 TSV 格式，列顺序为 query_id, doc_id, score

    返回：
        dict: {query_id: {doc_id: score, ...}, ...}
    """
    if isinstance(qrels_paths, str):
        qrels_paths = [qrels_paths]
    
    qrels = {}
    
    for path in qrels_paths:
        qrels_df = pd.read_csv(path, sep="\t", header=None, names=["query_id", "doc_id", "score"])
        for qid, group in qrels_df.groupby("query_id"):
            if qid not in qrels:
                qrels[qid] = {}
            qrels[qid].update(dict(zip(group["doc_id"], group["score"])))
    
    return qrels

In [63]:
def jsonl_results_loader(save_path,num_records=500):
    """加载 JSONL 格式的结果文件"""
    results = []
    with open(save_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line.strip())
            results.append(record)
    return results[0:num_records]

In [67]:
# results = jsonl_results_loader("/mnt/data1/workspace/zms/LeakDojo/results/fiqa/qwen3-32b/R__bge-large-en-v1_5_k10-RR__bge-reranker-large_n5-EX__bge-large-en-v1_5/WBTQ_RW-0_RR-0_EX-0_IF-0_OF-0_none_0_4ragas.jsonl",num_records=200) 
results = jsonl_results_loader("/mnt/data1/workspace/zms/LeakDojo/results/fiqa/qwen3-32b/R__bge-large-en-v1_5_k10-RR__bge-reranker-large_n5-EX__bge-large-en-v1_5/WBTQ_RW-0_RR-1_EX-0_IF-0_OF-0_none_0_4ragas.jsonl",num_records=200) 
retrieved_dict = {}

for item in results:
    qid = item["id"]  # 这里用 'id' 作为 query_id
    retrieved_dict[qid] = item["doc_ids"]  # 检索出的文档列表
# retrieved_dict

In [68]:
# 使用示例
qrels_path = ["data/fiqa/qrels/dev.tsv", "data/fiqa/qrels/test.tsv", "data/fiqa/qrels/train.tsv"]
qrels = load_qrels(qrels_path)

In [69]:
mean_ndcg(retrieved_dict, qrels, k=5)

0.18909139938686081